# Can we predict late delivery?
Follows `protocol.yaml`: temporal holdout from `holdout.csv`, 5-fold walk-forward CV on training rows, selection by mean CV ROC AUC, one holdout score.

**Feature choice:** `claim_paid_eur` is compensation on a claim settled *after* delivery, so it is not known at dispatch (prediction time) and leaks the outcome. It is excluded, along with `consignment` (identifier) and `dispatched_on` (split/CV ordering only).

In [ ]:
import json, hashlib
import numpy as np, pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score
from sklearn.base import clone

assert hashlib.sha256(open('data.csv','rb').read()).hexdigest()=='859a474044a292565d72b06a6180ade1c1adbc8c72a47c4a518186876f581f27'
assert hashlib.sha256(open('holdout.csv','rb').read()).hexdigest()=='fae5c02170970fe92fcadaee6d1109bb1d459c9dc033dfc487bb2c823d0f6736'
df=pd.read_csv('data.csv'); df['dispatched_on']=pd.to_datetime(df['dispatched_on'])
hold_idx=pd.read_csv('holdout.csv')['row'].to_numpy()
train_idx=np.setdiff1d(np.arange(len(df)),hold_idx)
# order training rows by time for walk-forward CV
train_idx=train_idx[np.argsort(df['dispatched_on'].to_numpy()[train_idx],kind='stable')]
print(len(train_idx),len(hold_idx), df.late_delivery.iloc[train_idx].mean(), df.late_delivery.iloc[hold_idx].mean())

In [ ]:
CAT=['carrier']; NUM=['distance_km','weight_kg','parcels']
FEATURES=CAT+NUM
X=df[FEATURES]; y=df['late_delivery']
def pre(): return ColumnTransformer([('c',OneHotEncoder(handle_unknown='ignore'),CAT),('n',StandardScaler(),NUM)])
# All candidates named before any is scored
CANDIDATES={
 'baseline_prior': Pipeline([('p',pre()),('m',DummyClassifier(strategy='prior'))]),
 'logreg_C1': Pipeline([('p',pre()),('m',LogisticRegression(C=1.0,max_iter=1000))]),
 'logreg_C0.1': Pipeline([('p',pre()),('m',LogisticRegression(C=0.1,max_iter=1000))]),
 'rf_depth6': Pipeline([('p',pre()),('m',RandomForestClassifier(n_estimators=300,max_depth=6,min_samples_leaf=20,random_state=0,n_jobs=-1))]),
 'gbm_depth2': Pipeline([('p',pre()),('m',GradientBoostingClassifier(n_estimators=150,max_depth=2,learning_rate=0.05,random_state=0))]),
}
print(list(CANDIDATES))

In [ ]:
tscv=TimeSeriesSplit(n_splits=5)
folds=[(train_idx[a],train_idx[b]) for a,b in tscv.split(train_idx)]
cv_scores={}
for name,est in CANDIDATES.items():
    s=[]
    for tr,va in folds:
        m=clone(est).fit(X.iloc[tr],y.iloc[tr])   # preprocessing fitted inside the fold
        s.append(float(roc_auc_score(y.iloc[va],m.predict_proba(X.iloc[va])[:,1])))
    cv_scores[name]=s
cvdf=pd.DataFrame({k:{'mean':np.mean(v),'std':np.std(v)} for k,v in cv_scores.items()}).T.sort_values('mean',ascending=False)
cvdf

In [ ]:
chosen=max(cv_scores,key=lambda k:np.mean(cv_scores[k]))
print('chosen by mean CV roc_auc:',chosen)
final=clone(CANDIDATES[chosen]).fit(X.iloc[train_idx],y.iloc[train_idx])
p=final.predict_proba(X.iloc[hold_idx])[:,1]
holdout_auc=float(roc_auc_score(y.iloc[hold_idx],p))   # scored once
print('HEADLINE holdout roc_auc:',round(holdout_auc,4))

In [ ]:
evidence={
 'features':FEATURES,
 'candidates':list(CANDIDATES),
 'cv_folds':[[tr.tolist(),va.tolist()] for tr,va in folds],
 'cv_scores':cv_scores,
 'chosen':chosen,
 'holdout_rows':hold_idx.tolist(),
 'holdout_scores':{chosen:holdout_auc},
}
json.dump(evidence,open('evidence.json','w'))

## Conclusion
See the CV table and headline holdout ROC AUC above. Only dispatch-time features (carrier, distance, weight, parcels) are used; the post-outcome claim amount is deliberately left out.